# The Newsvendor Problem

This notebook illustrates the Newsvendor problem, as presented in the first chapter of our course (see `01. Introduction`). We will formulate and solve the problem using two approaches: a purely mathematical resolution using its closed-form solution, and a stochastic programming formulation utilizing the `StochasticPrograms.jl` library. We will use Integer Programming to account for the discrete nature of the newspapers.

## Problem Description

A newsvendor must decide how many newspapers $x$ to buy each day to maximize their expected profit. The demand $\xi$ is uncertain and normally distributed.

**Parameters:**
- Cost per newspaper: $c = 0.15$
- Selling price per newspaper: $q = 0.25$
- Recourse/return price per unsold newspaper: $r = 0.02$
- Demand: $\xi \sim \mathcal{N}(650, 80^2)$ (mean 650, standard deviation 80)


## 1. Mathematical Resolution

The newsvendor wants to maximize the expected profit:
$$ \max_{x \geq 0} -cx + \mathcal{Q}(x) $$
where $\mathcal{Q}(x) = \mathbb{E}[Q(x, \xi)]$ is the expected return from selling newspapers given the demand $\xi$. The return $Q(x, \xi)$ is given by:

$$ Q(x, \xi) = \begin{cases} qx & \text{if } x \leq \xi \\ q\xi + r(x - \xi) & \text{if } x > \xi \end{cases} $$

Using the first-order optimality condition and taking the derivative of the expected profit with respect to $x$, we find that the optimal quantity $x^*$ satisfies:
$$ F(x^*) = \frac{q - c}{q - r} $$

Thus, $x^* = F^{-1} \left( \frac{q - c}{q - r} \right)$.
With our parameters, $\frac{q - c}{q - r} = \frac{0.25 - 0.15}{0.25 - 0.02} = \frac{0.10}{0.23} \approx 0.4348$.


In [ ]:
using Distributions

c = 0.15
q = 0.25
r = 0.02
mu = 650.0
sigma = 80.0

d = Normal(mu, sigma)
ratio = (q - c) / (q - r)

x_star = quantile(d, ratio)
println("Mathematical continuous solution: x* = ", x_star)
println("Mathematical integer solution: x* = ", round(Int, x_star))

## 2. Stochastic Programming Formulation

We can model this as a two-stage stochastic program. In the first stage, we decide the quantity $x$ of newspapers to purchase before knowing the demand. In the second stage, after observing the demand $\xi$, we determine how many newspapers we sell ($y$) and how many we return ($w$).

Since newspapers are discrete, we formulate this using Integer Programming variables.

**Note:** To be compatible with the latest library features, we will install the custom fork of `StochasticPrograms.jl` directly from GitHub.


In [ ]:
import Pkg
Pkg.add(url="https://github.com/fbastin/StochasticPrograms.jl")
Pkg.add("GLPK")
Pkg.add("MathOptInterface")

In [ ]:
using StochasticPrograms
using GLPK

# Define the scenario structure for the uncertain demand
@scenario NewsvendorScenario = begin
    xi::Int
end

# Formulate the two-stage stochastic model
@stochastic_model newsvendor_model begin
    @stage 1 begin
        @parameters c = 0.15
        @decision(model, x >= 0, Int)
        @objective(model, Max, -c * x)
    end
    @stage 2 begin
        @parameters q = 0.25
        @parameters r = 0.02
        @uncertain xi from NewsvendorScenario
        @variable(model, y >= 0, Int)
        @variable(model, w >= 0, Int)
        
        @objective(model, Max, q * y + r * w)
        @constraint(model, sell_limit, y <= xi)
        @constraint(model, balance, y + w == x)
    end
end

We now generate a set of scenarios by sampling the normal distribution and instantiate the model using the `GLPK` optimizer.

In [ ]:
# Generate 1000 demand scenarios
Random.seed!(42)
scenarios = [NewsvendorScenario(max(0, round(Int, rand(d)))) for _ in 1:1000]

# Instantiate and solve the model
sp = instantiate(newsvendor_model, scenarios, optimizer = GLPK.Optimizer)
optimize!(sp)

println("Optimal integer order quantity: ", optimal_decision(sp))
println("Expected profit: ", objective_value(sp))